In [0]:
# Data Loading 

buying_options_path = (
    "gs://market-intelligence-raw/ebay/"
    "item_details__buying_options/"
)

buying_options_df = (
    spark.read
    .option("multiLine", "false")
    .json(buying_options_path + "*.jsonl.gz")
)

print("Total rows:", buying_options_df.count())
print("Total columns:", len(buying_options_df.columns))

buying_options_df.printSchema()

In [0]:
# =====================================================
# Buying options cardinality per listing
# =====================================================
from pyspark.sql import functions as F

buying_option_counts = (
    buying_options_df
    .groupBy("_dlt_parent_id")
    .agg(
        F.count("*").alias("option_count")
    )
)

display(
    buying_option_counts
    .agg(
        F.count("*").alias("parent_count"),
        F.avg("option_count").alias("avg_options"),
        F.expr("percentile_approx(option_count, 0.25)").alias("p25"),
        F.expr("percentile_approx(option_count, 0.50)").alias("median"),
        F.expr("percentile_approx(option_count, 0.75)").alias("p75"),
        F.min("option_count").alias("min_options"),
        F.max("option_count").alias("max_options")
    )
)

In [0]:
# =====================================================
# Validate buying-options → Item Details relationship
# =====================================================

item_details_path = (
    "gs://market-intelligence-raw/ebay/item_details/"
)

item_details_df = (
    spark.read
    .option("multiLine", "false")
    .json(item_details_path + "*.jsonl.gz")
)



option_parent_ids = (
    buying_options_df
    .select(F.col("_dlt_parent_id").alias("parent_dlt_id"))
    .distinct()
)

item_parent_ids = (
    item_details_df
    .select(F.col("_dlt_id").alias("parent_dlt_id"))
    .distinct()
)

matched_parents = (
    option_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="inner")
    .count()
)

unmatched_parents = (
    option_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="left_anti")
    .count()
)

print("Buying-option parent IDs:", option_parent_ids.count())
print("Matched to current Item Details:", matched_parents)
print("Not found in current Item Details:", unmatched_parents)

In [0]:
# =====================================================
# Buying option values + data quality
# =====================================================

buying_options_df.groupBy("value").agg(
    F.count("*").alias("record_count"),
    F.countDistinct("_dlt_parent_id").alias("listing_count")
).orderBy(F.desc("record_count")).show(truncate=False)

buying_options_df.agg(
    F.sum(F.when(F.col("value").isNull(), 1).otherwise(0)).alias("null_value"),
    F.sum(
        F.when(
            F.col("value").isNotNull() & (F.trim(F.col("value")) == ""),
            1
        ).otherwise(0)
    ).alias("blank_value")
).show()

### Buying Options Profile

The `buying_options` dataset contains **87,736 records** across 4 columns:

- `_dlt_id`
- `_dlt_list_idx`
- `_dlt_parent_id`
- `value`

The table represents the available purchasing mechanism for individual listings, with _dlt_parent_id linking each record back to its parent Item Details listing.

# Buying Options per Listing
The current snapshot contains 62,982 parent listings with buying options.

| Metric | Result |
|---|---:|
| Parent listings with buying options | 62,982 |
| Average options per listing | 1.3930 |
| P25 | 1 |
| Median | 1 |
| P75 | 2 |
| Minimum options | 1 |
| Maximum options | 2 |

Most listings have a single buying option, while some listings have two. The median is 1 option per listing, and no listing has more than 2 options in the current snapshot.

The average of approximately 1.39 options per listing indicates that multiple purchasing mechanisms are present for a subset of listings.

# Buying Options → Item Details Relationship
The parent relationship was validated against the current Item Details dataset.

| Validation | Result |
|---|---:|
| Buying-option parent IDs | 62,982 |
| Matched to current Item Details | 62,982 |
| Not found in current Item Details | 0 |

All 62,982 parent IDs found in the buying-options dataset successfully match an existing Item Details record.

There are therefore no orphaned buying-option records in the current snapshot.

The relationship can be represented as:
```text
Item Details
    │
    │ 1
    │
    └──────────< Buying Options
                   many
```

# Buying Option Value Distribution
The value field contains three observed buying-option types:

| Buying Option | Record Count | Listing Count |
|---|---:|---:|
| `FIXED_PRICE` | 62,895 | 62,895 |
| `BEST_OFFER` | 23,738 | 23,738 |
| `AUCTION` | 1,103 | 1,103 |

FIXED_PRICE is the dominant buying option, accounting for approximately 71.7% of all buying-option records.

BEST_OFFER accounts for approximately 27.1%, while AUCTION represents approximately 1.3%.

The record_count and listing_count values are identical for each option type, indicating that each observed buying-option record corresponds to one parent listing for these values.

# Buying Option Data Quality
The value-level quality check found:

| Check | Result |
|---|---:|
| Null `value` records | 0 |
| Blank `value` records | 0 |
| Whitespace-only `value` records | 0 |

All buying-option records therefore contain a populated value.
The current data does not require missing-value handling for the value field.

Findings
The buying-options data has a relatively simple structure compared with the localized-aspects data.
Only three buying-option types are observed:

- `FIXED_PRICE`
- `BEST_OFFER`
- `AUCTION`

The data is fully connected to Item Details, with 0 unmatched parent IDs.

For the Silver layer, _dlt_parent_id can be used to associate buying options with the corresponding listing, while value can be retained as the normalized buying-option type.

The current snapshot therefore supports modeling buying options as a child/multi-valued relationship of Item Details, rather than adding multiple repeated buying-option columns to the main listing table.
